In [1]:
%run "./00_config.ipynb"

JAVA_HOME: C:\Java\jdk-17
java.exe found at: C:\Java\jdk-17\bin\java.EXE
PySpark home: c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark
bin dir exists: True
['beeline', 'beeline.cmd', 'docker-image-tool.sh', 'find-spark-home', 'find-spark-home.cmd', 'load-spark-env.cmd', 'load-spark-env.sh', 'pyspark', 'pyspark.cmd', 'pyspark2.cmd', 'run-example', 'run-example.cmd', 'spark-class', 'spark-class.cmd', 'spark-class2.cmd', 'spark-connect-shell', 'spark-shell', 'spark-shell.cmd', 'spark-shell2.cmd', 'spark-sql', 'spark-sql.cmd', 'spark-sql2.cmd', 'spark-submit', 'spark-submit.cmd', 'spark-submit2.cmd', 'sparkR', 'sparkR.cmd', 'sparkR2.cmd']
SPARK_HOME env: None
SPARK_HOME: None
JAVA_HOME  : C:\Java\jdk-17
HADOOP_HOME: C:\hadoop
winutils found at: C:\hadoop\bin\winutils.exe
Ready: C:\covid_pipeline\bronze
Ready: C:\covid_pipeline\silver
Ready: C:\covid_pipeline\gold
Spark version: 3.5.3
Spark master : local[*]
Parquet write test succeeded at: C:\covid_pipeline\

In [2]:
from pyspark.sql import functions as F

vacc = spark.read.parquet(SILVER_VACCINATION)
oxcgrt = spark.read.parquet(SILVER_OXCGRT)
population = spark.read.parquet(SILVER_POPULATION)
worldbank = spark.read.parquet(SILVER_WORLDBANK)
cases_deaths = spark.read.parquet(SILVER_CASES_DEATHS)  # already has iso_code + date resolved in 02

for name, df in [("vaccination", vacc), ("oxcgrt", oxcgrt), ("population", population),
                  ("worldbank", worldbank), ("cases_deaths", cases_deaths)]:
    print(f"{name}: {df.count()} rows, {len(df.columns)} cols")

vaccination: 84056 rows, 10 cols
oxcgrt: 7672 rows, 7 cols
population: 234 rows, 6 cols
worldbank: 218 rows, 5 cols
cases_deaths: 35156 rows, 8 cols


In [3]:
# Step 1: vaccination + oxcgrt on (iso_code, date) -- both daily time series
gold = (
    vacc.alias("v")
    .join(
        oxcgrt.select("iso_code", "date", "stringency_index", "govt_response_index",
                      "containment_health_index", "economic_support_index").alias("o"),
        on=["iso_code", "date"], how="left"
    )
)
print("After vaccination + oxcgrt join:", gold.count(), "rows")

After vaccination + oxcgrt join: 84056 rows


In [4]:
# Step 2: + cases_deaths, ALSO on (iso_code, date) now that it's a daily series (full_grouped.csv)
gold = gold.join(
    cases_deaths.select("iso_code", "date", "total_cases", "total_deaths",
                         "new_cases", "new_deaths", "case_fatality_rate"),
    on=["iso_code", "date"], how="left"
)
print("After cases_deaths join:", gold.count(), "rows")

After cases_deaths join: 84056 rows


In [5]:
# Step 3: + population (static, iso_code only)
gold = gold.join(
    population.select("iso_code", "continent", "population_2020"),
    on="iso_code", how="left"
)

# Step 4: + worldbank income classification (static, iso_code only)
gold_final = gold.join(
    worldbank.select("iso_code", "wb_region", "income_group"),
    on="iso_code", how="left"
)
print("Gold table final row count:", gold_final.count(), "| columns:", len(gold_final.columns))
gold_final.printSchema()

Gold table final row count: 84056 | columns: 23
root
 |-- iso_code: string (nullable = true)
 |-- date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- total_vaccinations: double (nullable = true)
 |-- people_vaccinated: double (nullable = true)
 |-- people_fully_vaccinated: double (nullable = true)
 |-- daily_vaccinations: double (nullable = true)
 |-- total_vaccinations_per_hundred: double (nullable = true)
 |-- vacc_per_100_people: double (nullable = true)
 |-- people_fully_vaccinated_per_hundred: double (nullable = true)
 |-- stringency_index: double (nullable = true)
 |-- govt_response_index: double (nullable = true)
 |-- containment_health_index: double (nullable = true)
 |-- economic_support_index: double (nullable = true)
 |-- total_cases: double (nullable = true)
 |-- total_deaths: double (nullable = true)
 |-- new_cases: double (nullable = true)
 |-- new_deaths: double (nullable = true)
 |-- case_fatality_rate: double (nullable = true)
 |-- continent: strin

In [6]:
# --- Data Quality Assessment (mirrors the report's Section H) ---
total_rows = gold_final.count()

null_vacc = gold_final.filter(F.col("vacc_per_100_people").isNull()).count()
print(f"CHECK 1 - Vaccination rate nulls: {null_vacc} / {total_rows} ({null_vacc/total_rows*100:.1f}%)")

matched_income = gold_final.filter(F.col("income_group").isNotNull()).select("iso_code").distinct().count()
total_countries = gold_final.select("iso_code").distinct().count()
print(f"CHECK 2 - Countries matched to income group: {matched_income} / {total_countries}")

dupes = gold_final.groupBy("iso_code", "date").count().filter("count > 1").count()
print(f"CHECK 3 - Duplicate iso_code+date combinations: {dupes}")

null_stringency = gold_final.filter(F.col("stringency_index").isNull()).count()
print(f"CHECK 4 - Stringency index nulls: {null_stringency} / {total_rows} ({null_stringency/total_rows*100:.1f}%)")

CHECK 1 - Vaccination rate nulls: 44746 / 84056 (53.2%)
CHECK 2 - Countries matched to income group: 205 / 217
CHECK 3 - Duplicate iso_code+date combinations: 0
CHECK 4 - Stringency index nulls: 80924 / 84056 (96.3%)


In [7]:
gold_final.write.mode("overwrite").parquet(GOLD_INTEGRATED)
print("Gold table written to:", GOLD_INTEGRATED)

Gold table written to: C:\covid_pipeline\gold\gold_integrated


In [8]:
# Optional: also export a CSV copy for quick manual inspection in Excel, alongside the Parquet (which is the source of truth)
gold_final.coalesce(1).write.mode("overwrite").option("header", True).csv(os.path.join(GOLD_DIR, "gold_final_csv"))
print("Also exported a CSV copy for manual inspection.")

Also exported a CSV copy for manual inspection.
